In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/ai_detector_project', exist_ok=True)
BACKUP_DIR = '/content/drive/MyDrive/ai_detector_project'
print("Backup dir ready:", BACKUP_DIR)

Mounted at /content/drive
Backup dir ready: /content/drive/MyDrive/ai_detector_project


In [ ]:
import os
csv_path_local = '/content/raid_balanced_final.csv'
csv_path_drive = os.path.join(BACKUP_DIR, 'raid_balanced_final.csv')

if os.path.exists(csv_path_drive):
    import shutil
    shutil.copy(csv_path_drive, csv_path_local)
    print("Loaded existing balanced dataset from Drive backup.")
else:
    print("No backup found on Drive yet — will collect fresh below.")


No backup found on Drive yet — will collect fresh below.


In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("liamdugan/raid", split="train", streaming=True)

domains_target = ['news', 'reddit', 'abstracts', 'recipes', 'reviews', 'poetry', 'books']
per_domain_ai = 3750
per_domain_human = 3750

ai_by_domain = {d: [] for d in domains_target}
human_by_domain = {d: [] for d in domains_target}

count = 0
for row in ds:
    count += 1
    d = row['domain']
    if d not in domains_target:
        continue
    if row['model'] == 'human':
        if len(human_by_domain[d]) < per_domain_human:
            human_by_domain[d].append(row)
    else:
        if len(ai_by_domain[d]) < per_domain_ai:
            ai_by_domain[d].append(row)

    if count % 100000 == 0:
        print(f"Processed {count} rows")
        print("AI:", {k: len(v) for k, v in ai_by_domain.items()})
        print("Human:", {k: len(v) for k, v in human_by_domain.items()})
        print("---")

    if all(len(v) >= per_domain_ai for v in ai_by_domain.values()) and \
       all(len(v) >= per_domain_human for v in human_by_domain.values()):
        break

ai_rows = [r for rows in ai_by_domain.values() for r in rows]
human_rows = [r for rows in human_by_domain.values() for r in rows]

final_df = pd.concat([pd.DataFrame(ai_rows), pd.DataFrame(human_rows)]).sample(frac=1, random_state=42).reset_index(drop=True)

print("\nLabel distribution:")
print(final_df['model'].apply(lambda x: 'human' if x=='human' else 'ai').value_counts())
print("\nDomain distribution:")
print(final_df['domain'].value_counts())

final_df.to_csv('/content/raid_balanced_final.csv', index=False)
print("\nSaved:", final_df.shape)

Processed 100000 rows
AI: {'news': 0, 'reddit': 0, 'abstracts': 3750, 'recipes': 0, 'reviews': 0, 'poetry': 0, 'books': 0}
Human: {'news': 0, 'reddit': 0, 'abstracts': 3750, 'recipes': 0, 'reviews': 0, 'poetry': 0, 'books': 0}
---
Processed 200000 rows
AI: {'news': 0, 'reddit': 0, 'abstracts': 3750, 'recipes': 0, 'reviews': 0, 'poetry': 0, 'books': 0}
Human: {'news': 0, 'reddit': 0, 'abstracts': 3750, 'recipes': 0, 'reviews': 0, 'poetry': 0, 'books': 0}
---
Processed 300000 rows
AI: {'news': 0, 'reddit': 0, 'abstracts': 3750, 'recipes': 0, 'reviews': 0, 'poetry': 0, 'books': 0}
Human: {'news': 0, 'reddit': 0, 'abstracts': 3750, 'recipes': 0, 'reviews': 0, 'poetry': 0, 'books': 0}
---
Processed 400000 rows
AI: {'news': 0, 'reddit': 0, 'abstracts': 3750, 'recipes': 0, 'reviews': 0, 'poetry': 0, 'books': 0}
Human: {'news': 0, 'reddit': 0, 'abstracts': 3750, 'recipes': 0, 'reviews': 0, 'poetry': 0, 'books': 0}
---
Processed 500000 rows
AI: {'news': 0, 'reddit': 0, 'abstracts': 3750, 'recip

In [ ]:
# احفظ على /content الأول (لو لسه معملتش)
final_df.to_csv('/content/raid_balanced_final.csv', index=False)

# وبعدين انسخها على Drive فورًا - ده اللي هيحميها من أي كراش
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/ai_detector_project', exist_ok=True)

import shutil
shutil.copy('/content/raid_balanced_final.csv', '/content/drive/MyDrive/ai_detector_project/raid_balanced_final.csv')

print("✅ Saved and backed up to Drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Saved and backed up to Drive


In [ ]:
print(final_df['model'].apply(lambda x: 'human' if x=='human' else 'ai').value_counts())
print(final_df['domain'].value_counts())

model
human    26250
ai       26250
Name: count, dtype: int64
domain
recipes      7500
reddit       7500
news         7500
poetry       7500
books        7500
reviews      7500
abstracts    7500
Name: count, dtype: int64


In [ ]:
final_df['label'] = final_df['model'].apply(lambda x: 0 if x == 'human' else 1)
final_df = final_df[['generation', 'label']].dropna()
final_df = final_df[final_df['generation'].str.strip().str.len() > 20]
final_df = final_df.drop_duplicates(subset='generation')
print(final_df.shape)
print(final_df['label'].value_counts())

# Backup للنسخة النضيفة كمان
final_df.to_csv(BACKUP_DIR + '/raid_preprocessed.csv', index=False)


(52338, 2)
label
1    26194
0    26144
Name: count, dtype: int64


In [ ]:
!pip install torchvision --index-url https://download.pytorch.org/whl/cu128 -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 76.7 MB/s eta 0:00:00


In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import torch
print(torch.__version__)

2.11.0+cu128


In [ ]:
!kaggle datasets download -d mkhaleddeaf/ai-detection-models-hc3 --unzip

Dataset URL: https://www.kaggle.com/datasets/mkhaleddeaf/ai-detection-models-hc3
License(s): CC0-1.0
100% 848M/848M [00:42<00:00, 20.9MB/s]



In [ ]:
mkdir -p ~/.kaggle && echo KGAT_53a5a18afc81358ad2b07c001246b398 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [ ]:
!pip uninstall torchvision -y

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128


In [ ]:
!ls -la

total 99512
drwxr-xr-x 1 root root      4096 Sep  3 04:12 .
drwxr-xr-x 1 root root      4096 Sep  3 02:54 ..
drwxr-xr-x 4 root root      4096 Aug 24 13:27 .config
drwxr-xr-x 2 root root      4096 Sep  3 04:12 deberta_main_detector
drwxr-xr-x 2 root root      4096 Sep  3 04:12 distilbert_final
drwx------ 5 root root      4096 Sep  3 03:24 drive
-rw-r--r-- 1 root root    161023 Sep  3 04:12 logreg_baseline.joblib
-rw-r--r-- 1 root root 100924433 Sep  3 04:09 raid_balanced_final.csv
drwxr-xr-x 1 root root      4096 Aug 24 13:28 sample_data
-rw-r--r-- 1 root root       209 Sep  3 04:12 step1_3_comparison.csv
-rw-r--r-- 1 root root    774040 Sep  3 04:12 tfidf_vectorizer.joblib


In [ ]:
import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd

final_df = pd.read_csv('/content/drive/MyDrive/ai_detector_project/raid_balanced_final.csv')

model_path = "./deberta_main_detector"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Model loaded on:", device)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Model loaded on: cuda


In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

train_df, val_df = train_test_split(
    final_df, test_size=0.1, stratify=final_df['label'], random_state=42
)
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True))

def tokenize_fn(batch):
    return tokenizer(batch['generation'], truncation=True, padding='max_length', max_length=512)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
train_ds = train_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print("Ready:", len(train_ds), "train /", len(val_ds), "val")

Map:   0%|          | 0/47104 [00:00<?, ? examples/s]

Map:   0%|          | 0/5234 [00:00<?, ? examples/s]

Ready: 47104 train / 5234 val


In [ ]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
    }

args = TrainingArguments(
    output_dir="./raid_continued_ft",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=1e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()



model.save_pretrained("./model_hc3_plus_raid")
tokenizer.save_pretrained("./model_hc3_plus_raid")



Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.004422,0.132472,0.975927,0.976369,0.959808,0.993511


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

NameError: name 'BACKUP_DIR' is not defined

In [ ]:
# 1) Mount Drive (لو لسه معملتش)
from google.colab import drive
drive.mount('/content/drive')

import os
BACKUP_DIR = '/content/drive/MyDrive/ai_detector_project'
os.makedirs(BACKUP_DIR, exist_ok=True)

# 2) احفظ الموديل والـ tokenizer محليًا الأول
model.save_pretrained("./model_hc3_plus_raid")
tokenizer.save_pretrained("./model_hc3_plus_raid")

# 3) انسخه على Drive
import shutil
final_model_backup = os.path.join(BACKUP_DIR, 'model_hc3_plus_raid')
if os.path.exists(final_model_backup):
    shutil.rmtree(final_model_backup)
shutil.copytree("./model_hc3_plus_raid", final_model_backup)

print("✅ Model backed up to:", final_model_backup)
print(os.listdir(final_model_backup))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model backed up to: /content/drive/MyDrive/ai_detector_project/model_hc3_plus_raid
['model.safetensors', 'config.json', 'tokenizer.json', 'tokenizer_config.json']


In [ ]:
results = trainer.evaluate()
print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.004422,0.132472,1,0.975927,0.976369,0.959808,0.993511


{'eval_loss': 0.1324719786643982, 'eval_accuracy': 0.9759266335498663, 'eval_f1': 0.9763690922730682, 'eval_precision': 0.9598082595870207, 'eval_recall': 0.9935114503816794}
